# Week 2, Day 1: Arrays: the shape of neural data

Companion notebook for the Engram lesson at https://engram-peteradepoju.vercel.app/learn/w002/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np

rates = np.array([12.0, 15.5, 9.8, 22.1])     # from a list
print(rates, rates.dtype, rates.shape)

zeros = np.zeros((3, 4))                       # 3 rows, 4 columns of zeros
t = np.arange(0, 1, 0.25)                      # 0, 0.25, 0.5, 0.75
grid = np.linspace(0, 1, 5)                    # 5 evenly spaced points including both ends
print(zeros.shape, t, grid)

# Arithmetic works element by element, with no loop.
print(rates * 2)
print(rates - rates.mean())

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
n_neurons, n_trials, n_bins = 5, 4, 8
data = rng.poisson(3, size=(n_neurons, n_trials, n_bins))   # spike counts

print("shape:", data.shape, " ndim:", data.ndim, " size:", data.size)
print("trial-averaged response of each neuron:", data.mean(axis=1).shape)
print("population activity per trial and bin:", data.mean(axis=0).shape)
print("total spikes per neuron:", data.sum(axis=(1, 2)))

In [ ]:
x = np.zeros((5, 4, 8))
print("strides in bytes:", x.strides, " (each float64 is 8 bytes)")
print("transposed shape:", x.T.shape, " strides:", x.T.strides)

In [ ]:
import time
x = np.random.default_rng(1).normal(size=1_000_000)

t0 = time.perf_counter()
total = 0.0
for v in x:
    total += v * v
loop_time = time.perf_counter() - t0

t0 = time.perf_counter()
total_np = np.sum(x * x)
vec_time = time.perf_counter() - t0

print(f"loop: {loop_time*1000:.1f} ms   vectorised: {vec_time*1000:.2f} ms   speed-up: {loop_time/vec_time:.0f}x")

### Exercise: Shapes of a session

A session has 30 neurons, 40 trials and 100 time bins of 10 ms. Create `session` with `rng.poisson(0.2, size=...)`. Compute `psth`, the trial-averaged counts for each neuron and bin, and `mean_rate`, each neuron's average firing rate in spikes per second (divide the mean count per bin by the bin width in seconds).

In [ ]:
import numpy as np
rng = np.random.default_rng(2)
session = ...
psth = ...
mean_rate = ...
print(session.shape, psth.shape, mean_rate.shape)
print("mean rates (Hz):", np.round(mean_rate[:5], 1))

In [ ]:
import numpy as np
def test_session():
    "session has shape (neurons, trials, bins)"
    assert session.shape == (30, 40, 100)

def test_psth():
    "psth is the trial-averaged response, shape (30, 100)"
    assert psth.shape == (30, 100)
    assert np.allclose(psth, session.mean(axis=1))

def test_rate():
    "mean_rate is one number per neuron, in spikes per second"
    assert mean_rate.shape == (30,)
    assert np.allclose(mean_rate, session.mean(axis=(1, 2)) / 0.01)

test_session()
test_psth()
test_rate()
print("All checks passed")

### Exercise: Remove the loop

Rewrite `zscore` without a loop. It should subtract the mean and divide by the standard deviation, for the whole array at once.

In [ ]:
import numpy as np

def zscore(x):
    m, s = x.mean(), x.std()
    out = np.empty_like(x)
    for i in range(len(x)):
        out[i] = (x[i] - m) / s
    return out

print(zscore(np.array([1.0, 2.0, 3.0])))

In [ ]:
import numpy as np
def test_same_answer():
    "zscore gives the same result as the loop version"
    x = np.random.default_rng(5).normal(3, 2, size=500)
    m, s = x.mean(), x.std()
    expected = np.array([(v - m) / s for v in x])
    assert np.allclose(zscore(x), expected)

def test_no_loop():
    "zscore has no for loop"
    import dis
    ops = {i.opname for i in dis.get_instructions(zscore)}
    assert "FOR_ITER" not in ops, "Use array operations only, with no loop or comprehension."

test_same_answer()
test_no_loop()
print("All checks passed")